In [30]:
%load_ext autoreload
%autoreload 2
import jax
import pgx
from pgx.experimental import auto_reset
from twentyfortyeight import *
import jax.numpy as jnp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
env = pgx.make("2048")
batch_size = 4096

init = jax.jit(jax.vmap(env.init))  # vectorize and JIT-compile
step = jax.jit(jax.vmap(auto_reset(env.step, env.init)))

key = jax.random.key(42)
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, batch_size)

state = init(keys)  # vectorized states
key, subkey = jax.random.split(key)
policy_network = MLP(496, 4, subkey)
key, subkey = jax.random.split(key)
value_network = MLP(496, 1, subkey)
obs_wrapper = ToInt(FlattenObservation())
policy_fn = make_policy_fn(policy_network, obs_wrapper)
next_state, current_obs, traj = collect_trajectory(step, state, state.observation, policy_fn, key, 1000)

In [37]:
traj.rewards.shape

(1000, 4096, 1)